In [2]:
!pip install pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 3.3 MB/s eta 0:00:00a 0:00:01


In [3]:
!pip install mistralai

  Using cached eval_type_backport-0.2.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached jsonpath_python-1.0.6-py3-none-any.whl.metadata (12 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.27.2-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.6 kB)
Using cached eval_type_backport-0.2.2-py3-none-any.whl (5.8 kB)
Using cached jsonpath_python-1.0.6-py3-none-any.whl (7.6 kB)
Using cached pydantic_core-2.27.2-cp310-cp310-macosx_11_0_arm64.whl (1.8 MB)
Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached h11-0.14.0-py3-none-any.whl (58 kB)


In [5]:
!pip install requests

  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.3-py3-none-any.whl (64 kB)


In [ ]:
from PIL import Image

import base64
import os
import requests
import pandas as pd
import time

from pydantic import BaseModel

from mistralai import Mistral
from utils import read_red_channel, read_green_channel, read_blue_channel, generate_random_sample

In [16]:
# Helper function
def encode_image(image_path):
    """Encode the image to base64."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {image_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None

# Specify model
model = "pixtral-12b-2409"

# Path to your image
image_path = "data/converted/19.jpg"

# Getting the base64 string
base64_image = encode_image(image_path)

api_key = os.environ["PIXTRAL"]
client = Mistral(api_key=api_key)

## Demo

In [ ]:
# Define the messages for the chat
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "For this task, act as a bioligist who is studying extrachromosomal DNA (ecDNA). Consider this metaphase image containing ecDNA. Analyze it for possible ecDNA."
            },
            {
                "type": "image_url",
                "image_url": f"data:image/jpeg;base64,{base64_image}" 
            }
        ]
    }
]

# Get the chat response
chat_response = client.chat.complete(
    model=model,
    messages=messages
)

# Print the content of the response
print(chat_response.choices[0].message.content)

In [94]:
def generate_response(client, model, img_path, prompt, context, response_format=None, temp=0.7):
    
    # Encode image
    img_input = encode_image(img_path)
    
    # Define the messages for the chat
    messages = [
        {
            "role":"system",
            "content": f'{context}'
            
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{prompt}"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{img_input}" 
                }
            ]
        }
    ]
    
    # Get the chat response
    chat_response = client.chat.complete(
        model=model,
        temperature=temp,
        messages=messages
    )

    # Return the content of the response
    return chat_response.choices[0].message.content

### Experiment #1: 0-Shot Learning

In [103]:
# Load the data
test_labels_path = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/test_im_ec_quantification.csv'
labels_df = pd.read_csv(test_labels_path, header=0, names=['img','ec'])
labels_df['img'] = labels_df['img'].apply(lambda x: x[:-4])
labels_df.head()

,img,ec
0,0,1
1,1,3
2,10,2
3,1000,2
4,1001,0


In [96]:
def pixtral_zero_shot(data_folder, runs):
    ec_predictions = []
    counter = 0
    
    context = """
            For this task, you will act as a pathologist who is studying 
            extrachromosomal DNA (ecDNA). You will be given multiple images
            and asked to identify the number of circular ecDNA structures.
            ecDNA is usually smaller than chromosomes or nuclei which are also present.
            """
    prompt = "Count the number of ecDNA in this image. Output only a single integer and no additional text."
    
    model = "pixtral-12b-2409"
    api_key = os.environ["PIXTRAL"]
    client = Mistral(api_key=api_key)
    
    for img in os.listdir(data_folder):
        if img.endswith('.png'):    
            print(img)
            
            img_path = os.path.join(data_folder, img)
            result = generate_response(client, model, img_path, prompt, context)
            ec_predictions.append(result)
            
            time.sleep(1)
            counter += 1

        if counter == runs:
            break
        
    return ec_predictions

In [104]:
test_folder = '/Users/mohit/Documents/GitHub/ecdna-analysis/test_im_data/labels'
sampled_path = './data/sampled'
sample_data = generate_random_sample(input_folder=test_folder, output_folder=sampled_path, sample_size=10)

preds = pixtral_zero_shot(sampled_path, 10)
preds

2379
2014.png
1849.png
849.png
1714.png
1854.png
743.png
1411.png
2050.png


['17', '12', '1', '5', '4', '2', '4', '15']

In [105]:
labels_df.head()

,img,ec
0,0,1
1,1,3
2,10,2
3,1000,2
4,1001,0


In [ ]:
for img in os.listdir('./data/sampled'):
    if img.endswith()
    labels_df[labels_df['img'].isin(preds)]

,img,ec
